# Experiment 3 — Target-aware E5 embeddings

MulTaBench’s pinned LoRA implementation trains one shared E5-small-v2 adapter per outer fold using training targets. Separate PCA32 and joint PCA64 feed the six fixed predictors. No held-out labels fit the encoder or PCA. The upstream 512-token/column-prefix policy differs from frozen Experiment 2; comparisons concern complete pipelines, not an isolated fine-tuning effect. Binary metrics use a fixed 0.5 threshold, with no inner encoder refits.


| View | Question |
|---|---|
| Coverage | Have all five fold-specific adapters and predictor fits completed? |
| TAR effect matrix | Does either prespecified recipe improve on the same predictor without text? |
| Frozen-E5 comparison | How does the whole TAR pipeline compare with frozen E5 at the same PCA budget? |
| Retained variance and compute | What compression and computational costs accompany the predictive effects? |

## 1. Scope and provenance

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib.pyplot as plt
from src.utils.notebook_display import display_local, display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.visualize import figures, style
style.apply()
NOTEBOOK = '6. experiment_3/01_target_aware_embeddings'
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope, provenance and saved evidence")
display_json("Notebook execution identity", provenance())
from src.evaluation.results import collect
from src.evaluation.notebook_study import planned_coverage, headline_summary, burden_table, contrast_summary
from src.visualize.study import coverage_figures, native_figures, burden_figures
from src.evaluation.endpoints import HEADLINES
status, folds, per_dataset, paired = collect()
summary = headline_summary(folds)
display_json("Saved result coverage", status)
display_json("Prespecified headline metrics", HEADLINES)
print("Only saved, validated folds are read. No fit, API call, download or hypothesis test runs here.")


Only saved, validated folds are read. No fit, API call, download or hypothesis test runs here.


## 2. Completion and fixed comparison grid

Each dataset/predictor has two TAR–PCA combinations. The shared non-text reference comes from Experiment 1.

In [2]:
report_section('2. Completion and fixed comparison grid')
coverage = planned_coverage(folds, "experiment_3")
display_table("Complete/planned embedding configurations", coverage, preview_rows=12)
for fig, name, caption in coverage_figures(coverage, "Experiment 3 configuration coverage"):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


dataset_id,model_id,planned,completed
BIN_TEXT_FAKE_JOB_POSTING,catboost,2,0
BIN_TEXT_FAKE_JOB_POSTING,kumo_tabular,2,0
BIN_TEXT_FAKE_JOB_POSTING,limix_2,2,0
BIN_TEXT_FAKE_JOB_POSTING,linear,2,0
BIN_TEXT_FAKE_JOB_POSTING,tabicl_v2,2,0
BIN_TEXT_FAKE_JOB_POSTING,tabpfn_3_5,2,0
BIN_TEXT_JIGSAW_TOXICITY,catboost,2,0
BIN_TEXT_JIGSAW_TOXICITY,kumo_tabular,2,0
BIN_TEXT_JIGSAW_TOXICITY,limix_2,2,0
BIN_TEXT_JIGSAW_TOXICITY,linear,2,0


## 3. Predictive effect of the two TAR–PCA recipes

Compare joint PCA64 and per-column PCA32 against the same predictor without text. Only complete five-fold arms enter the descriptive comparisons.

In [3]:
report_section('3. Predictive effect of the two TAR–PCA recipes')
from src.visualize.paired import effect_matrix_figures
display_table("All embedding recipe means and identities", summary[summary.phase.eq("experiment_3")])
for fig, name, caption in effect_matrix_figures(paired, "experiment_3"):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


arm_id,phase,dataset_id,task_type,model_id,checkpoint_id,feature_set,provider,encoder,pca_scope,pca_components,split_id,metric,mean,descriptive_fold_sd,defined_folds


### 3b. Matched frozen-E5 comparison

Pair TAR with frozen E5-small using the same predictor, split and PCA budget. This compares complete pipelines: upstream TAR also changes input formatting, truncation and coordinate standardization.

In [4]:
report_section("3b. Matched frozen-E5 comparison")
from src.evaluation.representation_views import tar_vs_frozen
tar_comparison = tar_vs_frozen(folds)
display_table("TAR minus frozen E5-small with the same predictor/PCA/split", contrast_summary(tar_comparison))
print("This compares complete pipelines: prefix, missing placeholders, token coverage and coordinate standardization also differ. AUC and R² do not depend on the binary threshold policy.")


This compares complete pipelines: prefix, missing placeholders, token coverage and coordinate standardization also differ. AUC and R² do not depend on the binary threshold policy.


## 4. What variance did training-only PCA retain?

Training data alone fits each projection and imputation map. Validation/test rows only transform through it. Explained variance is not task relevance.

In [5]:
report_section('4. What variance did training-only PCA retain?')
from src.evaluation.notebook_evidence import saved_pca_diagnostics
from src.visualize.paired import effect_matrix_figures, pca_variance_figures
pca = saved_pca_diagnostics(folds[folds.phase.eq("experiment_3")] if not folds.empty else folds)
display_table("Saved outer-training PCA diagnostics", pca)
for fig, name, caption in pca_variance_figures(pca):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


## 5. Compression versus measured compute

Compare actual column width and predictor fit time; model generation time remains a separate measured cost.

In [6]:
report_section('5. Compression versus measured compute')
burden = burden_table(folds, ["experiment_3"])
display_table("Embedding recipe runtime and width", burden)
for fig, name, caption in burden_figures(burden):
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


## Complete printed report

Everything displayed above is printed in section order. Individual predictions and the full metric store remain available in the referenced evaluation databases.

In [7]:
finish_report(NOTEBOOK, save.summary() + "\nNo fitting, inference or outcome-based method selection was performed.")

6. experiment_3/01_target_aware_embeddings

1. Scope, provenance and saved evidence


Notebook execution identity
{
  "code_sha256": "bd3e3f06da80877b51d7d281a32dadc634126b94a14de204d18bfa0abca5d98b",
  "git_head": "32c5d692626c2e202ec0fac8406034183bbb883a",
  "git_dirty": true,
  "tfm_library_pin": "286157a5f8a4b6c53cf1341309e81bd1ff4ce128 dm-library (remotes/origin/HEAD)\n 8f14e5945ca6b068ed0c76218e769d3e672d0d13 tfm-library (remotes/origin/HEAD)\n 914ef4e57e1f153080e2686d03179c223db38c9a vsc-library (remotes/origin/HEAD)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Saved result coverage
{
  "status": "no_completed_folds"
}


Prespecified headline metrics
{
  "binary_classification": "roc_auc",
  "multiclass_classification": "roc_auc_ovr_macro",
  "regression": "r2"
}


2. Completion and fixed comparison grid


Complete/planned embedding configurations
    